# Section 4: Master Manifest Construction
Builds a strict dataset contract locking the operational dataset without overwriting the reconciled authoritative raw manifest.


In [1]:
import pandas as pd
import os
from IPython.display import display

OUTPUT_ROOT = r"D:\SKIN CANCER/pipeline_output"
manifests_dir = os.path.join(OUTPUT_ROOT, "manifests")

raw_path = os.path.join(manifests_dir, "authoritative_raw_manifest.csv")
clean_path = os.path.join(manifests_dir, "cleaned_working_manifest.csv")
train_path = os.path.join(manifests_dir, "training_eligible_manifest.csv")


## Load Manifests


In [2]:
print("Loading raw and cleaned manifests...")
df_raw = pd.read_csv(raw_path)
df_clean = pd.read_csv(clean_path)
print(f"Raw manifest rows: {len(df_raw)}")
print(f"Cleaned manifest initial rows: {len(df_clean)}")


Loading raw and cleaned manifests...
Raw manifest rows: 20720
Cleaned manifest initial rows: 20720


## Normalize/Fix Schema


In [3]:
rename_map = {
    "matched_csv_image_id": "canonical_match_id",
    "review_status": "reconciliation_status",
    "readable_status": "file_accessible_status",
    "file_hash": "duplicate_hash_group"
}
# Only rename columns that exist
actual_renames = {k: v for k, v in rename_map.items() if k in df_clean.columns}
df_clean.rename(columns=actual_renames, inplace=True)

if "final_authoritative_label" not in df_clean.columns:
    df_clean["final_authoritative_label"] = df_clean["gt_label"]

legacy_cols = ["family_review_status", "family_review_decision", "remove_due_to_family_review_flag", "canonical_keep_flag"]
df_clean.drop(columns=[c for c in legacy_cols if c in df_clean.columns], inplace=True)

# Safely evaluate and set exclusion logic without blind force overrides natively
if "final_dataset_status" in df_clean.columns:
    if "final_exclusion_reason" in df_clean.columns:
        existing_reasons = set(df_clean[df_clean["final_dataset_status"] == "excluded"]["final_exclusion_reason"].dropna().unique())
        # If reasons are empty or only our duplicate family exclusion, enforce schema safely
        if len(existing_reasons) == 0 or existing_reasons == {"duplicate_family_permanently_excluded"} or existing_reasons == {""}:
            df_clean.loc[df_clean["final_dataset_status"] == "excluded", "final_exclusion_reason"] = "duplicate_family_permanently_excluded"
    else:
        df_clean["final_exclusion_reason"] = "none"
        df_clean.loc[df_clean["final_dataset_status"] == "excluded", "final_exclusion_reason"] = "duplicate_family_permanently_excluded"
        
    df_clean.loc[df_clean["final_dataset_status"] == "eligible", "final_exclusion_reason"] = "none"



## Build Cleaned Manifest


In [4]:
df_clean.to_csv(clean_path, index=False)
print(f"Cleaned working manifest generated and saved to {clean_path}.")
display(df_clean.head(2))


Cleaned working manifest generated and saved to D:\SKIN CANCER/pipeline_output\manifests\cleaned_working_manifest.csv.


,full_path,folder_name,raw_folder_label,file_name_with_extension,file_stem_raw,base_id_candidate,recognized_suffix,extension,file_exists,file_size_bytes,...,reconciliation_status,review_decision,review_notes,duplicate_hash_group,family_id,final_dataset_status,final_exclusion_reason,eligible_for_training,eligible_for_split,final_authoritative_label
0,D:\SKIN CANCER\DS\NV\ISIC_0000000.jpg,NV,NV,ISIC_0000000.jpg,ISIC_0000000,ISIC_0000000,NaN,.jpg,True,49964,...,provisionally_approved,NaN,NaN,153b10f7b8b82bf80badbfdf73a544312637a1950e2dad...,FAM_ISIC_0000000,eligible,none,True,True,NV
1,D:\SKIN CANCER\DS\NV\ISIC_0000001.jpg,NV,NV,ISIC_0000001.jpg,ISIC_0000001,ISIC_0000001,NaN,.jpg,True,38941,...,provisionally_approved,NaN,NaN,6180745ca3044c6267b58dd77ae821fca7df549c64bb65...,FAM_ISIC_0000001,eligible,none,True,True,NV


## Build Training Eligible Manifest


In [5]:
required_final_cols = [
    "final_dataset_status",
    "final_exclusion_reason",
    "eligible_for_training",
    "eligible_for_split",
    "final_authoritative_label",
    "full_path"
]

missing_required = [c for c in required_final_cols if c not in df_clean.columns]
if missing_required:
    raise ValueError(
        f"Cannot build training_eligible_manifest.csv because required columns are missing: {missing_required}"
    )

df_train = df_clean[
    (df_clean["final_dataset_status"] == "eligible") &
    (df_clean["eligible_for_training"] == True) &
    (df_clean["eligible_for_split"] == True)
].copy()

df_train.to_csv(train_path, index=False)

print(f"Training eligible manifest generated and saved to {train_path}.")
print(f"Training eligible row count: {len(df_train)}")
display(df_train.head(5))

Training eligible manifest generated and saved to D:\SKIN CANCER/pipeline_output\manifests\training_eligible_manifest.csv.
Training eligible row count: 20664


,full_path,folder_name,raw_folder_label,file_name_with_extension,file_stem_raw,base_id_candidate,recognized_suffix,extension,file_exists,file_size_bytes,...,reconciliation_status,review_decision,review_notes,duplicate_hash_group,family_id,final_dataset_status,final_exclusion_reason,eligible_for_training,eligible_for_split,final_authoritative_label
0,D:\SKIN CANCER\DS\NV\ISIC_0000000.jpg,NV,NV,ISIC_0000000.jpg,ISIC_0000000,ISIC_0000000,NaN,.jpg,True,49964,...,provisionally_approved,NaN,NaN,153b10f7b8b82bf80badbfdf73a544312637a1950e2dad...,FAM_ISIC_0000000,eligible,none,True,True,NV
1,D:\SKIN CANCER\DS\NV\ISIC_0000001.jpg,NV,NV,ISIC_0000001.jpg,ISIC_0000001,ISIC_0000001,NaN,.jpg,True,38941,...,provisionally_approved,NaN,NaN,6180745ca3044c6267b58dd77ae821fca7df549c64bb65...,FAM_ISIC_0000001,eligible,none,True,True,NV
2,D:\SKIN CANCER\DS\NV\ISIC_0000003.jpg,NV,NV,ISIC_0000003.jpg,ISIC_0000003,ISIC_0000003,NaN,.jpg,True,45774,...,provisionally_approved,NaN,NaN,e3092bd47d1955c117a18acb1e236222cae99acfbd393d...,FAM_ISIC_0000003,eligible,none,True,True,NV
3,D:\SKIN CANCER\DS\NV\ISIC_0000006.jpg,NV,NV,ISIC_0000006.jpg,ISIC_0000006,ISIC_0000006,NaN,.jpg,True,58570,...,provisionally_approved,NaN,NaN,755896e5f1593d8b705f3c54c9b2632c7ba894eb89c362...,FAM_ISIC_0000006,eligible,none,True,True,NV
4,D:\SKIN CANCER\DS\NV\ISIC_0000007.jpg,NV,NV,ISIC_0000007.jpg,ISIC_0000007,ISIC_0000007,NaN,.jpg,True,54175,...,provisionally_approved,NaN,NaN,2d3fe1d91afb6dcf7f21893ec8bfaf42b8ed885026bf88...,FAM_ISIC_0000007,eligible,none,True,True,NV


## Consistency Validations & Audit Printouts


In [6]:
print("=== CONSISTENCY VALIDATION ===")

check_1 = all(df_train["full_path"].isin(df_clean["full_path"]))
print(f"1. All rows in training exist in cleaned: {check_1}")

check_2 = all(df_clean["full_path"].isin(df_raw["full_path"]))
print(f"2. All rows in cleaned exist in raw: {check_2}")

check_3 = len(df_train[df_train["final_dataset_status"] == "excluded"]) == 0
print(f"3. No excluded row in training: {check_3}")

eligible_count = len(df_train)
print(f"4. Eligible row count: {eligible_count} (Expected 20664)")

excluded_count = len(df_clean[df_clean["final_dataset_status"] == "excluded"])
print(f"5. Excluded row count: {excluded_count} (Expected 56)")

class_counts = df_train["final_authoritative_label"].value_counts()
print(f"6. Final class counts:\n{class_counts.to_string()}")

check_7 = all(df_train["final_exclusion_reason"] == "none")
print(f"7. All eligible rows have final_exclusion_reason=none: {check_7}")


=== CONSISTENCY VALIDATION ===
1. All rows in training exist in cleaned: True
2. All rows in cleaned exist in raw: True
3. No excluded row in training: True
4. Eligible row count: 20664 (Expected 20664)
5. Excluded row count: 56 (Expected 56)
6. Final class counts:
final_authoritative_label
NV     12851
MEL     4504
BCC     3309
7. All eligible rows have final_exclusion_reason=none: True


## Save Summary & Schema Artifacts


In [7]:
report_data = {
    "total_reconciled_rows": len(df_clean),
    "total_eligible_rows": eligible_count,
    "total_excluded_rows": excluded_count,
    "excluded_reason": "duplicate_family_permanently_excluded",
    "excluded_reason_count": excluded_count,
    "eligible_NV_count": class_counts.get("NV", 0),
    "eligible_MEL_count": class_counts.get("MEL", 0),
    "eligible_BCC_count": class_counts.get("BCC", 0)
}

report_path = os.path.join(manifests_dir, "manifest_summary_report.csv")
pd.DataFrame([report_data]).to_csv(report_path, index=False)

schema_text = """Manifest Schema

full_path:
    Absolute file path to the image on disk.

folder_name:
    Folder where the image was discovered (NV, MEL, BCC).

file_name_with_extension:
    Original filename including extension.

file_stem_raw:
    Original filename without extension.

extension:
    File extension such as .jpg or .png.

base_id_candidate:
    Base identifier derived from the filename after approved suffix normalization logic.

recognized_suffix:
    Recognized suffix such as _downsampled when present.

canonical_match_id:
    Final canonical identifier matched to the ground truth CSV.

folder_label:
    Provisional disease label inferred from the parent folder.

gt_label:
    Authoritative disease label from the ground truth CSV.

final_authoritative_label:
    Final resolved label used by the pipeline. Must be NV, MEL, or BCC.

match_status:
    Match outcome between file identity and ground truth.
    Examples: matched_exact, matched_via_suffix_rule, unmatched, ambiguous_match.

label_agreement_status:
    Relationship between folder label and ground truth label.
    Examples: agree, disagree, missing_ground_truth.

age_approx:
    Approximate patient age from metadata CSV. Audit/reporting only in v1.

sex:
    Patient sex from metadata CSV. Audit/reporting only in v1.

anatom_site_general:
    Anatomical site from metadata CSV. Audit/reporting only in v1.

lesion_id:
    Lesion-level grouping identifier from metadata CSV. Split-support only in v1.

metadata_row_found:
    Boolean indicating whether metadata was matched for the row.

family_id:
    Group identifier for duplicate-family handling.

duplicate_hash_group:
    Exact duplicate content grouping identifier or hash-derived group label.

file_accessible_status:
    OS-level file accessibility status only. This is not image decode validity.

reconciliation_status:
    Final reconciliation state from source authority checks.

final_dataset_status:
    Final operational status of the row.
    Allowed values: eligible, excluded.

final_exclusion_reason:
    Final exclusion reason.
    Allowed values currently in this frozen state:
    - none
    - duplicate_family_permanently_excluded

eligible_for_training:
    Boolean indicating whether the row may enter model training.

eligible_for_split:
    Boolean indicating whether the row may enter train/validation/test split logic.

Operational rule:
    All downstream sections after Section 4 must use training_eligible_manifest.csv as the operational dataset source.
    Do not re-infer dataset state by rescanning folders.
"""

schema_path = os.path.join(manifests_dir, "manifest_schema.txt")
with open(schema_path, "w", encoding="utf-8") as f:
    f.write(schema_text)

print(f"Summary report saved to: {report_path}")
print(f"Schema file saved to: {schema_path}")
print("Summary Report CSV and Schema TXT securely flushed to disk.")

Summary report saved to: D:\SKIN CANCER/pipeline_output\manifests\manifest_summary_report.csv
Schema file saved to: D:\SKIN CANCER/pipeline_output\manifests\manifest_schema.txt
Summary Report CSV and Schema TXT securely flushed to disk.
